In [ ]:
import sys
sys.path.append('/home/apassi1/deepbottleneck')
import torch
import random
from calc_encoding import *
import xarray as xr
from torch_cv import *
from regression import *

In [ ]:
def compute_brainmaps(xtrain, xtest, ytrain, ytest, epsilon = 0.1, device='gpu'):
    
    n_samples = xtrain.shape[0] + xtest.shape[0]
    n_features = xtrain.shape[1]
    n_components = compute_johnson_lindenstrauss_limit(n_samples=n_samples, epsilon=epsilon)
    
    sparse_random_projection = SparseRandomProjection(
        n_components=n_components,
        density=None,
        seed=0,
        allow_expansion=False
    )
    
    xtrain = sparse_random_projection(xtrain)
    xtest = sparse_random_projection(xtest)

    xtrain = xtrain.cpu()
    xtest = xtest.cpu()
        
    ALPHA_RANGE = [10**i for i in range(10)]
    regression = TorchRidgeGCV(
        alphas=ALPHA_RANGE,
        fit_intercept=True,
        scale_X=False,
        scoring='pearsonr',
        store_cv_values=False,
        alpha_per_target=False,
        device=device
    )
    
    regression.fit(xtrain, ytrain)
    best_alpha = float(regression.alpha_)
    
    y_true, y_predicted = regression_shared_unshared(
        x_train=xtrain,
        x_test=xtest,
        y_train=ytrain,
        y_test=ytest,
        model=Ridge(alpha=best_alpha),
    )
    
    y_true = y_true.T
    y_predicted = y_predicted.T

    r2 = torch.stack([pearson_r(y_true_, y_predicted_)
                      for y_true_, y_predicted_ in zip(y_true, y_predicted)])
    return r2

In [ ]:
def generate_brainmaps(subjid, model_name, layerid, r2):
    
    dataset_path = f"/data/apassi1/for_atlas/general/preprocessed/subject={subjid}.nc"
    dataset = xr.open_dataarray(dataset_path)
    encoding_score = dataset[0].copy()
    encoding_score.values = r2
    
    save_path = f'/data/apassi1/brainmaps/subj{subjid}_{model_name}_{layerid}.nc'
    encoding_score.to_netcdf(save_path)

    print(f'Brain map saved to {save_path}')

In [ ]:
# initialise models
subjid = 0

conv_label = "new_imagenet-1k_rand"
boots_label = "new_imagenet-1k_pca"

conv_model = get_model(conv_label,model_type='trained_bottleneck',ckpt_epoch=2400)
boots_model = get_model(boots_label,model_type='trained_bottleneck',ckpt_epoch=2400)

models = [("conv_model", conv_model), ("boots_model", boots_model)]
layerids = [1, 3, 5, 7, 9, 11]

In [ ]:
du, ds = load_images(subjid)
ytrain, ytest = get_neural_activations(subjid, roi="general")

In [ ]:
for model_name, model in models:
    for layerid in layerids:
        print(layerid)
        xtrain, xtest = extract_activations(model, layerid, du, ds)
        xtrain, xtest = compute_srp(xtrain, xtest)
        xtrain = xtrain.cpu()
        xtest = xtest.cpu()
        r2 = compute_brainmaps(xtrain, xtest, ytrain, ytest, epsilon = 0.1, device='cuda')
        generate_brainmaps(subjid, model_name, layerid, r2)